# Random Walks and Geometric Brownian Motion

This notebook builds intuition for stochastic price paths, compares a simple discretisation with vectorised simulation, and examines how drift, volatility and random seeds affect simulated outcomes.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import risk_kit as rk

N_YEARS = 10
N_SCENARIOS = 100
MU = 0.07
SIGMA = 0.15
STEPS_PER_YEAR = 12
START_PRICE = 100
SEED = 42

## 1. Model

Under geometric Brownian motion, the continuously compounded price process follows:

$$dS_t = \mu S_t\,dt + \sigma S_t\,dW_t$$

The exact lognormal one-step gross return is $\exp[(\mu-\frac{1}{2}\sigma^2)\Delta t + \sigma\sqrt{\Delta t}Z]-1$, where $Z$ is standard normal.

In [ ]:
prices = rk.gbm(n_years=N_YEARS, n_scenarios=N_SCENARIOS, mu=MU, sigma=SIGMA,
                 steps_per_year=STEPS_PER_YEAR, s_0=START_PRICE, seed=SEED)
ax = prices.plot(figsize=(11, 6), legend=False, alpha=0.45)
ax.set(title='Geometric Brownian motion: simulated price paths', xlabel='Monthly time step', ylabel='Price index')
plt.tight_layout()
prices.iloc[[0, -1]].T.describe()

## 2. Simulated return distribution

The simulator can return simple returns directly. These are the inputs required by return-based backtests such as CPPI.

In [ ]:
simulated_returns = rk.gbm(n_years=N_YEARS, n_scenarios=N_SCENARIOS, mu=MU, sigma=SIGMA,
                             steps_per_year=STEPS_PER_YEAR, prices=False, seed=SEED)
ax = simulated_returns.stack().plot.hist(bins=50, figsize=(9, 5), edgecolor='white')
ax.set(title='Distribution of simulated monthly returns', xlabel='Monthly simple return')
plt.tight_layout()
simulated_returns.describe().T.head()

## 3. Parameter sensitivity

Higher volatility widens the distribution of possible outcomes. Drift shifts the expected growth rate but does not remove path uncertainty.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for sigma in [0.08, 0.15, 0.25]:
    paths = rk.gbm(n_years=10, n_scenarios=30, mu=0.07, sigma=sigma, s_0=100, seed=SEED)
    ax.plot(paths, alpha=0.25, linewidth=0.8)
ax.set(title='Effect of volatility on simulated paths', xlabel='Monthly time step', ylabel='Price index')
plt.tight_layout()

## 4. Reproducibility

A fixed seed makes a simulation repeatable. Changing the seed generates a different set of random paths while preserving the model assumptions.

In [ ]:
path_a = rk.gbm(n_years=2, n_scenarios=3, seed=7)
path_b = rk.gbm(n_years=2, n_scenarios=3, seed=7)
assert path_a.equals(path_b)
print('Reproducibility check passed.')

## Limitations

GBM assumes constant drift and volatility, continuous trading and lognormal price dynamics. It does not reproduce volatility clustering, jumps, fat tails or changing market regimes. Simulated paths are scenarios under assumptions, not forecasts.

## Saved output figures

The SVGs below are committed standalone previews. Figures marked illustrative are visual guides; rerun the analysis cells above to generate data-driven results from the notebook's inputs.

### Simulated price paths
![Simulated price paths](../outputs/04_gbm_price_paths.svg)

### Volatility comparison
![Volatility comparison](../outputs/04_gbm_volatility_comparison.svg)
